# Notebook 2: Fine-Tuning the Student Model

This notebook implements local-to-local distillation fine-tuning:

**Teacher Model → Student Model**

1. Load a smaller, quantized **student model** (e.g., Mistral-7B)
2. Set up QLoRA for memory-efficient fine-tuning
3. Load synthetic datasets generated by the teacher model
4. Train sequentially on each dataset with evaluation
5. Use rollback logic to keep only improvements
6. Track successful datasets with a bitmask

**Advantages of Local Distillation:**
- No API costs
- Full control over generation
- Reproducible results
- Privacy-preserving
- Unlimited training data generation

## Understanding Local Distillation

**What is Knowledge Distillation?**
- Transfer knowledge from a large "teacher" model to a smaller "student" model
- Student learns to mimic teacher's outputs, not just the original data
- Results in smaller, faster models with nearly the same performance

**Our Two-Stage Process:**

### Stage 1: Dataset Generation (`create_dataset.ipynb`)
- **Teacher Model** (70B): Generates high-quality synthetic training data
- Runs locally with 4-bit quantization
- Creates diverse prompt/completion pairs
- Self-judges quality and filters low-scoring examples

### Stage 2: Student Fine-Tuning (This Notebook)
- **Student Model** (7B): Learns from teacher's generated data
- Trained with QLoRA for efficiency
- Sequential training with rollback protection
- Results in a specialized, efficient model

**Benefits:**
- 10x smaller model with >90% of teacher performance
- 10x faster inference
- Can run on consumer GPUs
- No ongoing API costs

In [ ]:
# 1. Install Libraries
# 'accelerate' is key for multi-GPU and optimized loading
# 'bitsandbytes' is for 4-bit quantization
# 'peft' is for LoRA
# 'trl' is for the SFTTrainer
%pip install -q transformers peft bitsandbytes accelerate datasets trl

In [ ]:
# 2. Imports
import os
import glob
import random
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    logging
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer
from datasets import load_from_disk, load_dataset, Dataset

logging.set_verbosity_info()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 3. Configuration

# --- Choose your student model ---
# These are smaller, efficient models that will be fine-tuned
BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# Alternative student models:
# BASE_MODEL_NAME = "microsoft/Phi-3-medium-4k-instruct"
# BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

DATASET_DIR = "." # Directory where 'dataset_*' folders are saved
FINAL_MODEL_DIR = "./final_models"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print(f"Student Model: {BASE_MODEL_NAME}")
print(f"Dataset Directory: {DATASET_DIR}")
print(f"Output Directory: {FINAL_MODEL_DIR}")
print()

# 4-bit Quantization config (for student model during training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)
print("Student quantization: 4-bit NF4")

# LoRA Config (for student model)
lora_config = LoraConfig(
    r=16,                # Rank
    lora_alpha=32,       # Alpha (typically 2*r)
    lora_dropout=0.05,   # Dropout
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
    bias="none",
    task_type="CAUSAL_LM",
)
print("LoRA config: r=16, alpha=32, dropout=0.05")
print()

# Training Arguments (adjust per your GPU)
training_args = TrainingArguments(
    per_device_train_batch_size=2,        # Batch size per GPU
    gradient_accumulation_steps=4,        # Effective batch size = 2*4 = 8
    learning_rate=2e-4,                   # Learning rate
    logging_steps=10,                     # Log every N steps
    num_train_epochs=1,                   # 1 epoch per dataset file
    save_strategy="no",                   # We will save manually
    output_dir="./tmp_results",
    optim="paged_adamw_32bit",           # Memory-efficient optimizer
    fp16=False,                           # Don't use FP16 with QLoRA
    bf16=True,                            # Use bfloat16 for compute
    max_grad_norm=0.3,                    # Gradient clipping
    warmup_ratio=0.03,                    # Warmup steps
    group_by_length=True,                 # Group sequences by length
)
print("Training config:")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs per dataset: {training_args.num_train_epochs}")

print("\n" + "="*60)

## 4. Evaluation Function (Stub)

A real evaluation is complex. For this demo, we'll create a stub. You should replace this with a proper evaluation on a held-out test set, ideally using an LLM-as-a-Judge.

In [ ]:
# --- Create a small, held-out 'golden' test set ---
# These should be manually created to test key behaviors
GOLDEN_TEST_SET = [
    {"prompt": "...[USER'S SELECTION]... 'The rabbit-hole went straight on...' ...[USER QUESTION]... What happened to Alice?", "expected": "Alice fell down..."},
    {"prompt": "...[CONTEXT]... 'The Mock Turtle's story...' ...[USER QUESTION]... What was his story about?", "expected": "His story was about..."}
]
# In a real setup, you'd load this from a file.
eval_dataset = Dataset.from_list(GOLDEN_TEST_SET)

def evaluate_model(model, tokenizer):
    """
    Stub evaluation function.
    Generates responses for the golden set and returns a dummy score.
    TODO: Replace with LLM-as-a-Judge on the outputs.
    """
    print("Evaluating model...")
    model.eval() # Set model to evaluation mode
    
    try:
        # Example: Just check if the first prompt generates *anything*
        test_prompt = eval_dataset[0]['prompt']
        inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=50)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if len(response) > len(test_prompt):
            print(f"Test generation: {response[len(test_prompt):]}")
            # This is a placeholder. A real score would be computed here.
            # For demo, we'll return a random score to simulate improvement/regression.
            return random.uniform(0.5, 1.0) 
        else:
            return 0.0 # Failed generation
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return 0.0

## 5. Helper: Model Loader

This function loads the base model or merges LoRA adapters to create a new base model.

In [ ]:
def get_model_and_tokenizer(model_name_or_path):
    """Loads the model, applying 4-bit config and adding new LoRA adapters."""
    print(f"Loading model from: {model_name_or_path}")
    
    # Check if it's a PEFT (LoRA) model or a base model
    if os.path.exists(os.path.join(model_name_or_path, "adapter_config.json")):
        print("Found LoRA adapters. Loading and merging...")
        # Load the *original* base model first
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME, 
            quantization_config=bnb_config, 
            device_map="auto"
        )
        # Load and merge the adapters
        model = PeftModel.from_pretrained(base_model, model_name_or_path)
        model = model.merge_and_unload() # Merge adapters to create a new base model
        print("Merge complete.")
    else:
        # This is the first run, load the base model
        print("Loading base model...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path, 
            quantization_config=bnb_config, 
            device_map="auto" # Automatically uses all GPUs
        )
    
    # Add *new* LoRA adapters for the *next* training step
    model.config.use_cache = False # Required for gradient checkpointing
    model = get_peft_model(model, lora_config)
    
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    return model, tokenizer

## 6. The Bitmask Training Loop

In [ ]:
def format_dataset(batch):
    # Apply the chat template for fine-tuning
    # This assumes your dataset has 'prompt' and 'completion' columns
    texts = []
    for prompt, completion in zip(batch['prompt'], batch['completion']):
        texts.append(f"<s>[INST] {prompt} [/INST] {completion} </s>")
    return {"text": texts}

print("Starting sequential fine-tuning...")

dataset_files = sorted(glob.glob(f"{DATASET_DIR}/dataset_*"))
num_docs = len(dataset_files)
print(f"Found {num_docs} dataset files to process.")

best_model_path = BASE_MODEL_NAME  # Start with the original base model
trained_bitmask = 0

# Get initial score (before any training)
print("Evaluating base model...")
temp_model, temp_tok = get_model_and_tokenizer(BASE_MODEL_NAME)
best_model_score = evaluate_model(temp_model, temp_tok)
del temp_model, temp_tok # Clear VRAM
torch.cuda.empty_cache()
print(f"Base model score: {best_model_score:.4f}")

for i, doc_file_path in enumerate(dataset_files):
    current_doc_bit = 1 << i
    doc_name = os.path.basename(doc_file_path)
    print("-"*50)
    print(f"Training on Doc {i+1}/{num_docs}: {doc_name}")
    print(f"Loading base model from: {best_model_path}")
    
    # 1. Load the current best model
    model, tokenizer = get_model_and_tokenizer(best_model_path)
    
    # 2. Load the new dataset
    train_dataset = load_from_disk(doc_file_path)
    formatted_dataset = train_dataset.map(format_dataset, batched=True, remove_columns=['prompt', 'completion'])
    
    # 3. Setup Trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=formatted_dataset,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=2048, # Adjust as needed
        args=training_args,
    )
    
    # 4. Train
    print("Starting training...")
    trainer.train()
    print("Training complete.")
    
    # 5. Save the *new* LoRA adapters (not the whole model)
    new_mask = trained_bitmask | current_doc_bit
    new_model_path = f"{FINAL_MODEL_DIR}/model_mask_{new_mask:0{num_docs}b}"
    trainer.save_model(new_model_path)
    print(f"Saved new adapters to: {new_model_path}")
    
    # 6. Evaluate the newly trained model
    # We must merge it to evaluate it as a whole
    model = model.merge_and_unload()
    new_score = evaluate_model(model, tokenizer)
    print(f"New score: {new_score:.4f} (vs. Best: {best_model_score:.4f})")

    # 7. Rollback Logic
    if new_score > best_model_score:
        print(f"**IMPROVEMENT**: Keeping changes. New best model is {new_model_path}")
        best_model_path = new_model_path
        best_model_score = new_score
        trained_bitmask = new_mask
    else:
        print(f"**REGRESSION**: Discarding changes. Rolling back to {best_model_path}")
        # We do nothing; best_model_path and trained_bitmask remain the same
        # The model we just saved (new_model_path) will be ignored
    
    # 8. Clean up VRAM
    del model, tokenizer, trainer, train_dataset, formatted_dataset
    torch.cuda.empty_cache()

print("="*50)
print("Sequential Fine-Tuning Finished.")
print(f"Final best model adapters saved at: {best_model_path}")
print(f"Successful document bitmask: {trained_bitmask:0{num_docs}b}")

## 7. Merging Final Model (Optional)

After the loop, `best_model_path` points to the LoRA adapters. You can merge these into the base model for a final, single model directory for easy inference.

In [ ]:
print("Loading base model for final merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Loading best adapters from {best_model_path}...")
model = PeftModel.from_pretrained(base_model, best_model_path)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

final_merged_path = f"{FINAL_MODEL_DIR}/final_merged_model_mask_{trained_bitmask:0{num_docs}b}"
print(f"Saving final merged model to {final_merged_path}...")
model.save_pretrained(final_merged_path)
tokenizer.save_pretrained(final_merged_path)
print("Done!")